In [1]:
import duckdb

# Convert via a single SQL query
duckdb.query("COPY (SELECT * FROM 'credit_card_default_ssb_dataset_50000.csv') TO 'credit_card_default_ssb_dataset_50000.parquet' (FORMAT PARQUET)")


In [3]:
from Experimental_Codes.ssb_experimental import *
import pyarrow.parquet as pq
import pyarrow.csv as csv

# Example Feature Groupings Setup
business_groups = {
"delinquency_vars":["dpd_avg_3m", "dpd_avg_6m", "dpd_avg_12m","max_dpd_12m"],            
"transaction_vars":["txn_count_avg_3m", "txn_count_avg_6m", "txn_count_avg_12m"],
"spend_vars":["spend_avg_3m", "spend_avg_6m", "spend_avg_12m"],
"repayment_vars":["payment_ratio_avg_3m","payment_ratio_avg_6m","payment_ratio_avg_12m"],
"card_utilization_vars":["utilization_avg_3m","utilization_avg_6m","utilization_avg_12m","utilization_max_12m"],
}


# Initialize the builder
builder = StrategicSegmentBuilder(
    target="default_flag",
    n_jobs=-1,
    min_sample_size=5000,
    min_lift=1.5,
    top_n_vars=30,
    max_segments=10,
    enable_diversity=True,
    max_feature_reuse=1,
    enable_1way=True,  # Include 1-way rules from final output
    enable_2way=True,  # Exclude 2-way rules from final output
    enable_3way=True,  # Exclude 3-way rules from final output
    feature_groups=business_groups,
    ignore_features=None
)

# 2. Define your dynamically scaled optimization grid
my_search_grid = {
    "min_sample_size": [10000, 5000,500,100],
    "min_lift": [2.0,1.5]
}

data = pq.read_table(r"/workspaces/Strategic_Segment_Builder/credit_card_default_ssb_dataset_50000.parquet")

segments_df = builder.extract_segments(data, my_search_grid)
final_eval = builder.evaluate_final_coverage(data)

2026-07-03 15:23:56,226 | INFO     | [ssb_experimental.py:113] | Feature group validation passed. (17 features mapped)
2026-07-03 15:23:56,231 | INFO     | [ssb_experimental.py:317] | Iteration 1 | Remaining Volume: 50,000 | Base Rate: 7.28%


2026-07-03 15:24:04,135 | INFO     | [ssb_experimental.py:393] | No active candidates cleared criteria pool across grid variations. Stopping.


In [2]:
from prettytable import PrettyTable
table = PrettyTable()
table.field_names = list(final_eval.columns)
for _, row in final_eval.iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+--------------------+--------------------+---------------------+--------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate |     capture_rate    |        lift        |
+---------+-------------+---------------+--------------------+--------------------+---------------------+--------------------+
|   0.0   |   48738.0   |     3052.0    | 6.262054249251098  |       7.284        |        97.476       | 0.8596999243892227 |
|   1.0   |    151.0    |     123.0     | 81.45695364238411  |       7.284        |        0.302        | 11.182997479734228 |
|   2.0   |    105.0    |      72.0     | 68.57142857142857  |       7.284        |         0.21        | 9.413979759943516  |
|   3.0   |    133.0    |      79.0     |  59.3984962406015  |       7.284        |        0.266        |  8.15465352012651  |
|   4.0   |    118.0    |      70.0     | 59.32203389830509  |       7.284        | 0.23600000000000002 |  8.14

In [3]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in segments_df.iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   utilization_avg_3m=[0.85, inf) & payment_ratio_avg_3m=(-inf, 0.30) & max_dpd_12m=[33.62, inf)
SQL Filter: utilization_avg_3m >= 0.85 AND payment_ratio_avg_3m < 0.30 AND max_dpd_12m >= 33.62
--------------------------------------------------
Segment ID: 2
Raw Rule:   dpd_avg_12m=[16.05, inf) & bureau_score=[582.50, 617.50) & payment_ratio_avg_6m=(-inf, 0.37)
SQL Filter: dpd_avg_12m >= 16.05 AND (bureau_score >= 582.50 AND bureau_score < 617.50) AND payment_ratio_avg_6m < 0.37
--------------------------------------------------
Segment ID: 3
Raw Rule:   dpd_avg_3m=[30.32, inf) & utilization_avg_6m=[0.74, inf) & missed_payments_last_6m=[2.50, inf)
SQL Filter: dpd_avg_3m >= 30.32 AND utilization_avg_6m >= 0.74 AND missed_payments_last_6m >= 2.50
--------------------------------------------------
Segment ID: 4
Raw Rule:   risk_segment=<StringArray>
['HighRisk']
Length: 1, dtype: str & utilization_avg_12m=[0.66, inf) & annual_income=[20292

In [4]:
final_eval

,segment,total_count,target_events,response_rate,base_response_rate,capture_rate,lift
0,0,48738,3052.0,6.262054,7.284,97.476,0.859700
1,1,151,123.0,81.456954,7.284,0.302,11.182997
2,2,105,72.0,68.571429,7.284,0.210,9.413980
3,3,133,79.0,59.398496,7.284,0.266,8.154654
4,4,118,70.0,59.322034,7.284,0.236,8.144156
5,5,112,60.0,53.571429,7.284,0.224,7.354672
6,6,128,49.0,38.281250,7.284,0.256,5.255526
7,7,120,41.0,34.166667,7.284,0.240,4.690646
8,8,108,30.0,27.777778,7.284,0.216,3.813533
9,9,287,66.0,22.996516,7.284,0.574,3.157127


In [5]:
# Preparing the dataset for scoring and decile banding.
conn = duckdb.connect()
predicted = conn.register("predicted", data)
predicted = conn.query("""
    SELECT *, 
    CASE WHEN utilization_avg_3m >= 0.85 AND payment_ratio_avg_3m < 0.30 AND max_dpd_12m >= 33.62 THEN 1 ELSE 0 END AS seg_1,
    CASE WHEN dpd_avg_12m >= 16.05 AND (bureau_score >= 582.50 AND bureau_score < 617.50) AND payment_ratio_avg_6m < 0.37 THEN 1 ELSE 0 END AS seg_2,
    CASE WHEN dpd_avg_3m >= 30.32 AND utilization_avg_6m >= 0.74 AND missed_payments_last_6m >= 2.50 THEN 1 ELSE 0 END AS seg_3,
    CASE WHEN risk_segment IN ('HighRisk') AND utilization_avg_12m >= 0.66 AND (annual_income >= 202928.00 AND annual_income < 261094.00) THEN 1 ELSE 0 END AS seg_4,
    CASE WHEN (spend_avg_3m >= 5510.50 AND spend_avg_3m < 7765.50) AND dpd_avg_6m >= 20.74 AND payment_ratio_avg_12m < 0.45 THEN 1 ELSE 0 END AS seg_5,
    CASE WHEN num_delinquent_accounts >= 2.50 AND (utilization_max_12m >= 0.76 AND utilization_max_12m < 0.86) AND (cash_advance_amt_3m >= 12229.00 AND cash_advance_amt_3m < 16168.50)
 THEN 1 ELSE 0 END AS seg_6,
    CASE WHEN txn_count_avg_12m < 11.50 AND (spend_avg_6m >= 8366.50 AND spend_avg_6m < 11895.50) AND num_open_trades >= 8.50 THEN 1 ELSE 0 END AS seg_7,                  
    CASE WHEN spend_avg_12m < 4538.50 AND txn_count_avg_6m < 10.87 AND (total_credit_limit >= 77194.00 AND total_credit_limit < 107939.50)
 THEN 1 ELSE 0 END AS seg_8,       
    CASE WHEN age < 21.50 AND txn_count_avg_3m < 9.71 THEN 1 ELSE 0 END AS seg_9,    

    FROM predicted
""").df()
conn.close()

In [10]:
scorer = StrategicSegmentScore(
    target_col="default_flag",
    primary_key="index",
    segment_cols=["seg_1", "seg_2", "seg_3", "seg_4", "seg_5", "seg_6", "seg_7", "seg_8", "seg_9"],
)

In [11]:
scorer.calculate_and_export_weights(predicted, export_path="scorecard_model_test1.json")

2026-07-03 15:09:43,139 | INFO     | [SSB.py:512] | Initializing DuckDB scorecard engine for 50,000 records...
2026-07-03 15:09:43,176 | INFO     | [SSB.py:546] | Computing harmonic scorecard weights...
2026-07-03 15:09:43,177 | INFO     | [SSB.py:583] | Scoring training dataset via NumPy Linear Algebra engine...
2026-07-03 15:09:43,180 | INFO     | [SSB.py:594] | Dataset Zero-Inflation Rate: 92.72%
2026-07-03 15:09:43,181 | INFO     | [SSB.py:597] | High Zero-Inflation (>=80%). Isolating Active Population...
2026-07-03 15:09:43,181 | INFO     | [SSB.py:609] | Calibrating deciles across 1,262 target customers...


{'model_metadata': {'total_training_population': 50000,
  'active_scored_population': 1262,
  'active_population_pct': 2.52,
  'baseline_event_rate': 0.0728},
 'segment_weights': {'seg_1': {'weight': 73,
   'lift': 11.183,
   'response_rate': 0.8146,
   'capture_rate': 0.0338},
  'seg_2': {'weight': 40,
   'lift': 9.3872,
   'response_rate': 0.6838,
   'capture_rate': 0.022},
  'seg_3': {'weight': 41,
   'lift': 8.5569,
   'response_rate': 0.6233,
   'capture_rate': 0.025},
  'seg_4': {'weight': 39,
   'lift': 8.4709,
   'response_rate': 0.617,
   'capture_rate': 0.0239},
  'seg_5': {'weight': 37,
   'lift': 8.2179,
   'response_rate': 0.5986,
   'capture_rate': 0.0233},
  'seg_6': {'weight': 17,
   'lift': 5.6298,
   'response_rate': 0.4101,
   'capture_rate': 0.0157},
  'seg_7': {'weight': 10,
   'lift': 4.5047,
   'response_rate': 0.3281,
   'capture_rate': 0.0115},
  'seg_8': {'weight': 7,
   'lift': 4.0945,
   'response_rate': 0.2982,
   'capture_rate': 0.0093},
  'seg_9': {'weigh

In [12]:
scorecard_json_path = "/workspaces/Strategic_Segment_Builder/scorecard_model_test1.json"
logging.info(f"Loading scorecard model artifact from {scorecard_json_path}...")
with open(scorecard_json_path, "r") as json_file:
    model_artifact = json.load(json_file)

2026-07-03 15:09:57,789 | INFO     | [2422015793.py:2] | Loading scorecard model artifact from /workspaces/Strategic_Segment_Builder/scorecard_model_test1.json...


In [13]:
model_artifact.get("decile_min_thresholds")

{'1': 73,
 '2': 41,
 '3': 40,
 '4': 39,
 '5': 21,
 '6': 17,
 '7': 14,
 '8': 14,
 '9': 10,
 '10': 7}

In [14]:
for key, value in model_artifact.get("segment_weights").items():
    print(f"Segment: {key} | Weight: {value['weight']}")

Segment: seg_1 | Weight: 73
Segment: seg_2 | Weight: 40
Segment: seg_3 | Weight: 41
Segment: seg_4 | Weight: 39
Segment: seg_5 | Weight: 37
Segment: seg_6 | Weight: 17
Segment: seg_7 | Weight: 10
Segment: seg_8 | Weight: 7
Segment: seg_9 | Weight: 14


In [15]:
model_artifact.get("decile_min_thresholds")

{'1': 73,
 '2': 41,
 '3': 40,
 '4': 39,
 '5': 21,
 '6': 17,
 '7': 14,
 '8': 14,
 '9': 10,
 '10': 7}

In [12]:
conn = duckdb.connect()
scored = conn.register("scored", predicted)
scored = conn.query("""
WITH CTE AS (
    SELECT *, 
    CASE WHEN seg_1 = 1 THEN 32 ELSE 0 END AS seg_1_weighted,
    CASE WHEN seg_2 = 1 THEN 38 ELSE 0 END AS seg_2_weighted,
    CASE WHEN seg_3 = 1 THEN 56 ELSE 0 END AS seg_3_weighted,
    CASE WHEN seg_4 = 1 THEN 44 ELSE 0 END AS seg_4_weighted,
    CASE WHEN seg_5 = 1 THEN 65 ELSE 0 END AS seg_5_weighted,
    CASE WHEN seg_6 = 1 THEN 53 ELSE 0 END AS seg_6_weighted,
    CASE WHEN seg_7 = 1 THEN 73 ELSE 0 END AS seg_7_weighted,
    CASE WHEN seg_8 = 1 THEN 76 ELSE 0 END AS seg_8_weighted,
    CASE WHEN seg_9 = 1 THEN 78 ELSE 0 END AS seg_9_weighted,
    CASE WHEN seg_10 = 1 THEN 58 ELSE 0 END AS seg_10_weighted,
    FROM scored),
    CTE2 AS (
    SELECT *, (seg_1_weighted + seg_2_weighted + seg_3_weighted + seg_4_weighted + seg_5_weighted+ seg_6_weighted+ seg_7_weighted+ seg_8_weighted+ seg_9_weighted+ seg_10_weighted) AS total_weight
                     FROM CTE)
SELECT *, CASE WHEN total_weight >=348 THEN 1
                    WHEN total_weight >= 292 THEN 2
                    WHEN total_weight >= 193 THEN 3
                    WHEN total_weight >= 111 THEN 4
                    WHEN total_weight >= 53 THEN 5
                    WHEN total_weight >= 0 THEN 6
                    WHEN total_weight >= 0 THEN 7
                    WHEN total_weight >= 0 THEN 8
                    WHEN total_weight >= 0 THEN 9
                    WHEN total_weight >= 0 THEN 10
                    ELSE 0 END AS decile_band
                    
                     FROM CTE2
""").to_df()
conn.close()

In [13]:
conn = duckdb.connect()
scored = conn.register("scored", scored)
scored = conn.query("""SELECT decile_band, 
                    COUNT(*) AS count, 
                    SUM(target_cc_take) AS events, 
                    (SUM(target_cc_take)*100.0/COUNT(*)) AS response_rate
FROM scored
GROUP BY decile_band
ORDER BY decile_band
""").to_df()
conn.close()

In [14]:
scored

,decile_band,count,events,response_rate
0,1,497157,360542.0,72.520753
1,2,148889,90160.0,60.555179
2,3,341872,209145.0,61.176405
3,4,226810,121119.0,53.401085
4,5,300092,140590.0,46.848966
5,6,1485180,428444.0,28.847951
